# 📊 FMCG InsightAI — Sales, Profit & Inventory Intelligence System

**Author:** Manthan Tighare  
**Dataset:** Indian FMCG Retail Sales, Customer & Inventory (2024)  
**Records:** 100,000 transactions · 21 columns · Jan 2024 – Dec 2024  
**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn · Plotly · Scikit-learn  

---

## Table of Contents
1. [Environment Setup & Imports](#1)
2. [Data Loading & First Look](#2)
3. [Data Quality Audit](#3)
4. [Data Cleaning & Preprocessing](#4)
5. [Part 1 — Sales & Profit Intelligence](#5)
6. [Part 2 — Inventory & Customer Intelligence](#6)
7. [Part 3 — AI Business Insights](#7)
8. [Key Business Findings](#8)

<a id='1'></a>
## 1. Environment Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from pathlib import Path

# ── Notebook display settings ─────────────────────────────────────────────────
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.width', 120)

plt.rcParams.update({
    'figure.facecolor': '#0e0e1a',
    'axes.facecolor':   '#1e1e2e',
    'axes.edgecolor':   '#2e2e4e',
    'axes.labelcolor':  '#c8cfe8',
    'text.color':       '#c8cfe8',
    'xtick.color':      '#a0a8c0',
    'ytick.color':      '#a0a8c0',
    'grid.color':       '#1e1e2e',
    'axes.grid':        True,
    'font.family':      'sans-serif',
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})

PALETTE = ['#7c5cd8','#3b82d4','#10b981','#f59e0b','#ef4444','#06b6d4','#ec4899','#84cc16']
sns.set_palette(PALETTE)

print('All libraries imported successfully.')

<a id='2'></a>
## 2. Data Loading & First Look

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────────
DATA_PATH = Path('../data/fmcg_data.csv')
df_raw = pd.read_csv(DATA_PATH)

print(f'Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
df_raw.head(3)

In [ ]:
# ── Column data types ─────────────────────────────────────────────────────────
df_raw.dtypes

In [ ]:
# ── Statistical summary (numeric columns) ────────────────────────────────────
df_raw[['Units','Cost_Price','Selling_Price','Revenue','Cost',
        'Margin','Margin_%','Stock_On_Hand','Reorder_Level',
        'Lead_Time_Days','Customer_Age']].describe().round(2)

In [ ]:
# ── Categorical unique values ─────────────────────────────────────────────────
for col in ['City','Store_Format','Category','Brand','Channel','Payment_Mode','Customer_Gender']:
    print(f'{col:20s} ({df_raw[col].nunique()} unique): {sorted(df_raw[col].dropna().unique().tolist())}')

In [ ]:
# ── Date range ────────────────────────────────────────────────────────────────
dates = pd.to_datetime(df_raw['Invoice_Date'], errors='coerce')
print(f'Date range: {dates.min()} → {dates.max()}')
print(f'Loyalty_Flag distribution:\n{df_raw["Loyalty_Flag"].value_counts().to_string()}')

<a id='3'></a>
## 3. Data Quality Audit

In [ ]:
# ── Missing values ────────────────────────────────────────────────────────────
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]
print('Columns with missing values:')
print(missing_df.to_string())

In [ ]:
# ── Duplicate records ─────────────────────────────────────────────────────────
print(f'Fully duplicate rows      : {df_raw.duplicated().sum()}')
print(f'Duplicate Invoice_ID      : {df_raw["Invoice_ID"].duplicated().sum()}')

In [ ]:
# ── Computed column validation ────────────────────────────────────────────────
checks = {
    'Revenue = Units × Selling_Price' : (df_raw['Revenue'] - df_raw['Units']*df_raw['Selling_Price']).abs().max(),
    'Cost    = Units × Cost_Price'    : (df_raw['Cost']    - df_raw['Units']*df_raw['Cost_Price']).abs().max(),
    'Margin  = Revenue − Cost'        : (df_raw['Margin']  - (df_raw['Revenue']-df_raw['Cost'])).abs().max(),
    'Margin_% = Margin / Revenue'     : (df_raw['Margin_%']- df_raw['Margin']/df_raw['Revenue']).abs().max(),
}
for label, err in checks.items():
    status = '✅ PASS' if err < 1e-6 else '❌ FAIL'
    print(f'{status}  {label:<38}  max_err = {err:.2e}')

In [ ]:
# ── Negative / zero price / unit checks ──────────────────────────────────────
for col in ['Units','Cost_Price','Selling_Price','Revenue','Cost']:
    n = (df_raw[col] <= 0).sum()
    status = '✅' if n == 0 else '⚠️'
    print(f'{status}  {col:20s}  values <= 0: {n}')

print(f'\n✅  Selling_Price < Cost_Price: {(df_raw["Selling_Price"] < df_raw["Cost_Price"]).sum()} rows')

In [ ]:
# ── Visualise missing values ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 3))
bars = ax.barh(['Customer_Age','Customer_Gender'], [40.08, 5.05], color=['#ef4444','#f59e0b'])
ax.set_xlabel('Missing %', color='#c8cfe8')
ax.set_title('Missing Values by Column', color='#f0f2ff')
for bar, val in zip(bars, [40.08, 5.05]):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val}%', va='center', color='#c8cfe8', fontsize=11)
ax.set_xlim(0, 50)
plt.tight_layout()
plt.show()

<a id='4'></a>
## 4. Data Cleaning & Preprocessing

In [ ]:
df = df_raw.copy()

# ── 1. Parse Invoice_Date ─────────────────────────────────────────────────────
df['Invoice_Date'] = pd.to_datetime(df['Invoice_Date'], errors='coerce')
df['Month']        = df['Invoice_Date'].dt.month
df['Month_Label']  = df['Invoice_Date'].dt.strftime('%b')
df['Quarter']      = df['Invoice_Date'].dt.quarter
df['DayOfWeek']    = df['Invoice_Date'].dt.day_name()
print('✅  Invoice_Date parsed')

# ── 2. Remove duplicate Invoice IDs ──────────────────────────────────────────
before = len(df)
df = df.drop_duplicates(subset='Invoice_ID', keep='first').reset_index(drop=True)
print(f'✅  Duplicate Invoice IDs removed: {before - len(df)} rows dropped ({len(df):,} remain)')

# ── 3. Impute Customer_Age with median ────────────────────────────────────────
age_median = df['Customer_Age'].median()
df['Customer_Age'] = df['Customer_Age'].fillna(age_median)
print(f'✅  Customer_Age imputed with median ({age_median:.0f} yrs)')

# ── 4. Fill Customer_Gender missing as Unknown ────────────────────────────────
df['Customer_Gender'] = df['Customer_Gender'].fillna('Unknown')
print('✅  Customer_Gender: missing filled as "Unknown"')

# ── 5. Create Inventory_Status ───────────────────────────────────────────────
df['Inventory_Status'] = np.where(
    df['Stock_On_Hand'] <= df['Reorder_Level'],
    'Reorder Required',
    'Sufficient Stock'
)
reorder_n = (df['Inventory_Status'] == 'Reorder Required').sum()
print(f'✅  Inventory_Status created  →  Reorder Required: {reorder_n:,} ({reorder_n/len(df)*100:.1f}%)')

# ── Final check ───────────────────────────────────────────────────────────────
print(f'\nFinal dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Remaining missing values: {df.isnull().sum().sum()}')

In [ ]:
# ── Correlation matrix (numeric features) ────────────────────────────────────
num_cols = ['Units','Revenue','Cost','Margin','Margin_%',
            'Stock_On_Hand','Reorder_Level','Lead_Time_Days','Customer_Age']
corr = df[num_cols].corr().round(3)

fig, ax = plt.subplots(figsize=(10, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            ax=ax, linewidths=0.5, linecolor='#0e0e1a',
            annot_kws={'size': 9}, vmin=-1, vmax=1)
ax.set_title('Correlation Matrix — Numeric Features', pad=14)
plt.tight_layout()
plt.show()

<a id='5'></a>
## 5. Part 1 — Sales & Profit Intelligence

In [ ]:
# ── Helper ────────────────────────────────────────────────────────────────────
def fmt_inr(v):
    if v >= 1e7:  return f'₹{v/1e7:,.2f} Cr'
    if v >= 1e5:  return f'₹{v/1e5:,.2f} L'
    if v >= 1e3:  return f'₹{v/1e3:,.1f} K'
    return f'₹{v:,.2f}'

# ── KPIs ──────────────────────────────────────────────────────────────────────
total_rev      = df['Revenue'].sum()
total_units    = df['Units'].sum()
total_margin   = df['Margin'].sum()
avg_margin_pct = df['Margin_%'].mean() * 100
total_txn      = len(df)

print('=' * 55)
print(f'  💰  Total Revenue        : {fmt_inr(total_rev)}')
print(f'  📦  Total Units Sold     : {total_units:,}')
print(f'  📊  Total Margin         : {fmt_inr(total_margin)}')
print(f'  📐  Avg Margin %%         : {avg_margin_pct:.2f}%%')
print(f'  🧾  Total Transactions   : {total_txn:,}')
print('=' * 55)

In [ ]:
# ── 5.1  Monthly Revenue & Margin Trend ──────────────────────────────────────
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly = (
    df.groupby('Month_Label', sort=False)
    .agg(Revenue=('Revenue','sum'), Margin=('Margin','sum'))
    .reset_index()
)
monthly['Month_Label'] = pd.Categorical(monthly['Month_Label'], categories=month_order, ordered=True)
monthly = monthly.sort_values('Month_Label')

fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()

bars = ax1.bar(monthly['Month_Label'], monthly['Revenue'], color='#7c5cd8', alpha=0.85, label='Revenue')
ax2.plot(monthly['Month_Label'], monthly['Margin'], color='#10b981', linewidth=2.5,
         marker='o', markersize=6, label='Margin')

ax1.set_ylabel('Revenue (₹)', color='#7c5cd8')
ax2.set_ylabel('Margin (₹)',  color='#10b981')
ax1.tick_params(axis='y', labelcolor='#7c5cd8')
ax2.tick_params(axis='y', labelcolor='#10b981')
ax1.set_title('Monthly Revenue & Margin Trend — 2024')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left',
           facecolor='#1e1e2e', edgecolor='#2e2e4e', labelcolor='#c8cfe8')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.2  Revenue by Category ──────────────────────────────────────────────────
cat_rev = df.groupby('Category')['Revenue'].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors = plt.cm.get_cmap('viridis', len(cat_rev))(np.linspace(0.3, 0.9, len(cat_rev)))
bars = ax.barh(cat_rev.index, cat_rev.values, color=PALETTE[:len(cat_rev)])
for bar, val in zip(bars, cat_rev.values):
    ax.text(val * 1.005, bar.get_y() + bar.get_height()/2,
            fmt_inr(val), va='center', fontsize=9, color='#c8cfe8')
ax.set_xlabel('Total Revenue (₹)')
ax.set_title('Revenue by Category')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.3  Top Brands by Revenue + Avg Margin % by Brand ───────────────────────
brand_rev    = df.groupby('Brand')['Revenue'].sum().sort_values(ascending=False)
brand_margin = df.groupby('Brand')['Margin_%'].mean().mul(100).sort_values(ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.bar(brand_rev.index, brand_rev.values, color=PALETTE)
for i, (brand, val) in enumerate(brand_rev.items()):
    ax1.text(i, val * 1.005, fmt_inr(val), ha='center', fontsize=8, color='#c8cfe8')
ax1.set_title('Top Brands by Total Revenue')
ax1.set_ylabel('Revenue (₹)')
ax1.tick_params(axis='x', rotation=15)

bars2 = ax2.bar(brand_margin.index, brand_margin.values, color=PALETTE)
for bar, val in zip(bars2, brand_margin.values):
    ax2.text(bar.get_x() + bar.get_width()/2, val * 1.005,
             f'{val:.1f}%', ha='center', fontsize=9, color='#c8cfe8')
ax2.set_title('Avg Margin % by Brand')
ax2.set_ylabel('Avg Margin %')
ax2.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# ── 5.4  Revenue by City ──────────────────────────────────────────────────────
city_rev = df.groupby('City')['Revenue'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(city_rev.index, city_rev.values, color=PALETTE)
for bar, val in zip(bars, city_rev.values):
    ax.text(bar.get_x() + bar.get_width()/2, val * 1.005,
            fmt_inr(val), ha='center', fontsize=9, color='#c8cfe8')
ax.set_title('Total Revenue by City')
ax.set_ylabel('Revenue (₹)')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.5  Revenue by Channel (pie) ─────────────────────────────────────────────
ch_rev = df.groupby('Channel')['Revenue'].sum()

fig, ax = plt.subplots(figsize=(6, 5))
wedges, texts, autotexts = ax.pie(
    ch_rev.values, labels=ch_rev.index, autopct='%1.1f%%',
    colors=['#7c5cd8','#3b82d4','#10b981'],
    wedgeprops=dict(edgecolor='#0e0e1a', linewidth=2),
    startangle=140
)
for t in texts + autotexts:
    t.set_color('#c8cfe8')
ax.set_title('Revenue Share by Sales Channel')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.6  Revenue vs Margin scatter by Category ────────────────────────────────
cat_summary = (
    df.groupby('Category')
    .agg(Revenue=('Revenue','sum'), Margin=('Margin','sum'), Units=('Units','sum'))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(
    cat_summary['Revenue'], cat_summary['Margin'],
    s=cat_summary['Units'] / 50,
    c=PALETTE[:len(cat_summary)], alpha=0.85, edgecolors='#2e2e4e', linewidths=1
)
for _, row in cat_summary.iterrows():
    ax.annotate(row['Category'], (row['Revenue'], row['Margin']),
                textcoords='offset points', xytext=(6, 4),
                color='#c8cfe8', fontsize=9)
ax.set_xlabel('Total Revenue (₹)')
ax.set_ylabel('Total Margin (₹)')
ax.set_title('Revenue vs Margin by Category  (bubble size = units sold)')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.7  Avg Margin % by Category ────────────────────────────────────────────
cat_marg = df.groupby('Category')['Margin_%'].mean().mul(100).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(cat_marg.index, cat_marg.values, color=PALETTE[:len(cat_marg)])
for bar, val in zip(bars, cat_marg.values):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9, color='#c8cfe8')
ax.set_xlabel('Avg Margin %')
ax.set_title('Profitability: Avg Margin % by Category')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.8  High-Revenue & High-Margin Segment Table ─────────────────────────────
seg = (
    df.groupby(['Category','Brand'])
    .agg(
        Total_Revenue =('Revenue','sum'),
        Total_Margin  =('Margin','sum'),
        Avg_Margin_Pct=('Margin_%','mean'),
        Total_Units   =('Units','sum')
    )
    .reset_index()
    .sort_values('Total_Revenue', ascending=False)
    .head(15)
)
seg['Total_Revenue']    = seg['Total_Revenue'].map(fmt_inr)
seg['Total_Margin']     = seg['Total_Margin'].map(fmt_inr)
seg['Avg_Margin_Pct']   = seg['Avg_Margin_Pct'].map(lambda v: f'{v*100:.1f}%')
seg['Total_Units']      = seg['Total_Units'].map(lambda v: f'{v:,}')
seg.index = range(1, len(seg)+1)
print('Top 15 High-Revenue & High-Margin Segments:')
seg

<a id='6'></a>
## 6. Part 2 — Inventory & Customer Intelligence

In [ ]:
# ── 6.1  Inventory KPIs ──────────────────────────────────────────────────────
reorder_req = (df['Inventory_Status'] == 'Reorder Required').sum()
sufficient  = (df['Inventory_Status'] == 'Sufficient Stock').sum()
avg_soh     = df['Stock_On_Hand'].mean()
avg_lead    = df['Lead_Time_Days'].mean()

print('=' * 55)
print(f'  ⚠️  Reorder Required     : {reorder_req:,}  ({reorder_req/len(df)*100:.1f}%)')
print(f'  ✅  Sufficient Stock     : {sufficient:,}  ({sufficient/len(df)*100:.1f}%)')
print(f'  📊  Avg Stock on Hand    : {avg_soh:,.0f} units')
print(f'  🚚  Avg Lead Time        : {avg_lead:.1f} days')
print('=' * 55)


In [ ]:
# ── 6.2  Inventory Status: Split & by Category ───────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

status_counts = df['Inventory_Status'].value_counts()
wedges, texts, autotexts = ax1.pie(
    status_counts.values, labels=status_counts.index, autopct='%1.1f%%',
    colors=['#10b981', '#ef4444'],
    wedgeprops=dict(edgecolor='#0e0e1a', linewidth=2, width=0.5),
    startangle=90,
)
for t in texts + autotexts:
    t.set_color('#c8cfe8')
ax1.set_title('Inventory Status Split')

cat_status = df.groupby(['Category', 'Inventory_Status']).size().unstack(fill_value=0)
cat_status.plot(kind='bar', ax=ax2,
                color={'Sufficient Stock': '#10b981', 'Reorder Required': '#ef4444'},
                stacked=True, edgecolor='#0e0e1a')
ax2.set_title('Inventory Status by Category')
ax2.set_ylabel('Transactions')
ax2.tick_params(axis='x', rotation=20)
ax2.legend(facecolor='#1e1e2e', edgecolor='#2e2e4e', labelcolor='#c8cfe8')

plt.tight_layout()
plt.show()


In [ ]:
# ── 6.3  Avg Stock vs Reorder Level by Brand ─────────────────────────────────
brand_inv = (
    df.groupby('Brand')
    .agg(Avg_Stock=('Stock_On_Hand', 'mean'), Avg_Reorder=('Reorder_Level', 'mean'))
    .sort_values('Avg_Stock', ascending=False)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 5))
x = range(len(brand_inv))
ax.bar(x, brand_inv['Avg_Stock'], color='#3b82d4', label='Avg Stock on Hand', alpha=0.85)
ax.plot(x, brand_inv['Avg_Reorder'], color='#ef4444', linewidth=2.5,
        marker='D', markersize=7, linestyle='--', label='Avg Reorder Level')
ax.set_xticks(list(x))
ax.set_xticklabels(brand_inv['Brand'], rotation=10)
ax.set_ylabel('Units')
ax.set_title('Avg Stock on Hand vs Avg Reorder Level by Brand')
ax.legend(facecolor='#1e1e2e', edgecolor='#2e2e4e', labelcolor='#c8cfe8')
plt.tight_layout()
plt.show()


In [ ]:
# ── 6.4  Lead Time Distribution & Avg Lead Time by Brand ─────────────────────
brand_lt = df.groupby('Brand')['Lead_Time_Days'].mean().sort_values(ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(df['Lead_Time_Days'], bins=12, color='#7c5cd8', edgecolor='#0e0e1a', alpha=0.9)
ax1.set_xlabel('Lead Time (Days)')
ax1.set_ylabel('Frequency')
ax1.set_title('Lead Time Distribution')

bars = ax2.bar(brand_lt.index, brand_lt.values, color=PALETTE)
for bar, val in zip(bars, brand_lt.values):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.05,
             f'{val:.1f}d', ha='center', fontsize=9, color='#c8cfe8')
ax2.set_ylabel('Avg Lead Time (Days)')
ax2.set_title('Avg Lead Time by Brand')
ax2.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()


In [ ]:
# ── 6.5  Reorder Required by City ────────────────────────────────────────────
city_reorder = (
    df[df['Inventory_Status'] == 'Reorder Required']
    .groupby('City').size().sort_values()
)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(city_reorder.index, city_reorder.values, color='#ef4444')
for bar, val in zip(bars, city_reorder.values):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9, color='#c8cfe8')
ax.set_xlabel('Reorder Required Count')
ax.set_title('Reorder Required Transactions by City')
plt.tight_layout()
plt.show()


### 👥 Customer Intelligence

In [ ]:
# ── 6.6  Customer KPIs ───────────────────────────────────────────────────────
loyal_rev    = df[df['Loyalty_Flag'] == 1]['Revenue'].sum()
nonloyal_rev = df[df['Loyalty_Flag'] == 0]['Revenue'].sum()
loyal_pct    = df['Loyalty_Flag'].mean() * 100
avg_age      = df['Customer_Age'].mean()

print('=' * 55)
print(f'  🌟  Loyal Customers      : {loyal_pct:.1f}% of all transactions')
print(f'  💎  Loyal Revenue        : {fmt_inr(loyal_rev)}')
print(f'  👤  Non-Loyal Revenue    : {fmt_inr(nonloyal_rev)}')
print(f'  🎂  Avg Customer Age     : {avg_age:.1f} years')
print('=' * 55)


In [ ]:
# ── 6.7  Loyal vs Non-Loyal Revenue + Customer Gender ───────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Loyalty donut
wedges, texts, autotexts = ax1.pie(
    [loyal_rev, nonloyal_rev],
    labels=['Loyal', 'Non-Loyal'],
    autopct='%1.1f%%',
    colors=['#f59e0b', '#3b82d4'],
    wedgeprops=dict(edgecolor='#0e0e1a', linewidth=2, width=0.52),
    startangle=140,
)
for t in texts + autotexts:
    t.set_color('#c8cfe8')
ax1.set_title('Revenue: Loyal vs Non-Loyal')

# Gender bar
gender_map = {'M': 'Male', 'F': 'Female', 'O': 'Other', 'Unknown': 'Unknown'}
df['Gender_Label'] = df['Customer_Gender'].map(gender_map).fillna('Unknown')
gender_counts = df['Gender_Label'].value_counts()
bars = ax2.bar(gender_counts.index, gender_counts.values,
               color=['#7c5cd8', '#ec4899', '#3b82d4', '#a0a8c0'])
for bar, val in zip(bars, gender_counts.values):
    ax2.text(bar.get_x() + bar.get_width()/2, val * 1.005,
             f'{val:,}', ha='center', fontsize=9, color='#c8cfe8')
ax2.set_title('Transactions by Customer Gender')
ax2.set_ylabel('Transactions')

plt.tight_layout()
plt.show()


In [ ]:
# ── 6.8  Revenue by Age Group + Loyal vs Non-Loyal Units by Category ─────────
df['Age_Group'] = pd.cut(
    df['Customer_Age'],
    bins=[17, 25, 35, 45, 55, 65],
    labels=['18-25', '26-35', '36-45', '46-55', '56-64']
)
age_rev = df.groupby('Age_Group', observed=True)['Revenue'].sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

bars = ax1.bar(age_rev.index.astype(str), age_rev.values, color=PALETTE)
for bar, val in zip(bars, age_rev.values):
    ax1.text(bar.get_x() + bar.get_width()/2, val * 1.005,
             fmt_inr(val), ha='center', fontsize=8, color='#c8cfe8')
ax1.set_title('Revenue by Customer Age Group')
ax1.set_ylabel('Revenue (₹)')

loy_cat = df.groupby(['Category', 'Loyalty_Flag'])['Units'].sum().unstack()
loy_cat.columns = ['Non-Loyal', 'Loyal']
loy_cat.plot(kind='bar', ax=ax2, color=['#3b82d4', '#f59e0b'], edgecolor='#0e0e1a')
ax2.set_title('Loyal vs Non-Loyal Units by Category')
ax2.set_ylabel('Units Sold')
ax2.tick_params(axis='x', rotation=20)
ax2.legend(facecolor='#1e1e2e', edgecolor='#2e2e4e', labelcolor='#c8cfe8')

plt.tight_layout()
plt.show()


In [ ]:
# ── 6.9  Channel Preference + Payment Mode ───────────────────────────────────
ch_counts  = df.groupby('Channel').size().sort_values(ascending=False)
pay_counts = df['Payment_Mode'].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

bars = ax1.bar(ch_counts.index, ch_counts.values, color=['#7c5cd8', '#3b82d4', '#10b981'])
for bar, val in zip(bars, ch_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, val * 1.005,
             f'{val:,}', ha='center', fontsize=9, color='#c8cfe8')
ax1.set_title('Customer Channel Preference')
ax1.set_ylabel('Transactions')

wedges, texts, autotexts = ax2.pie(
    pay_counts.values, labels=pay_counts.index, autopct='%1.1f%%',
    colors=['#7c5cd8', '#3b82d4', '#10b981', '#f59e0b'],
    wedgeprops=dict(edgecolor='#0e0e1a', linewidth=2, width=0.48),
    startangle=140,
)
for t in texts + autotexts:
    t.set_color('#c8cfe8')
ax2.set_title('Payment Mode Breakdown')

plt.tight_layout()
plt.show()


In [ ]:
# ── 6.10  Customer Revenue Summary by City & Channel ─────────────────────────
cust_tbl = (
    df.groupby(['City', 'Channel'])
    .agg(Transactions=('Invoice_ID', 'count'),
         Total_Revenue=('Revenue', 'sum'),
         Loyal_Count=('Loyalty_Flag', 'sum'))
    .reset_index()
    .sort_values('Total_Revenue', ascending=False)
    .head(20)
)
cust_tbl['Loyal_%']       = (cust_tbl['Loyal_Count'] / cust_tbl['Transactions'] * 100).map(lambda v: f'{v:.1f}%')
cust_tbl['Total_Revenue'] = cust_tbl['Total_Revenue'].map(fmt_inr)
cust_tbl.index = range(1, len(cust_tbl) + 1)
print('Top 20 City × Channel combinations by Revenue:')
cust_tbl


<a id='7'></a>
## 7. Part 3 — AI Business Insights

### Technique Selection Rationale
| Approach | Decision |
|---|---|
| Supervised Classification | ❌ REJECTED — no labelled fraud/churn column |
| Time Series Forecasting | ❌ REJECTED — one full year, no future holdout |
| **Isolation Forest (Anomaly Detection)** | ✅ SELECTED — unsupervised, detects unusual financial & inventory patterns |
| **K-Means Clustering** | ✅ SELECTED — segments 100K transactions into actionable business profiles |


### 7.1 Isolation Forest — Anomaly Detection

**Features used:** Revenue, Units, Margin_%, Selling_Price, Cost_Price, Stock_On_Hand, Reorder_Level, Lead_Time_Days  
**Contamination:** 3%  ·  **n_estimators:** 200  ·  **random_state:** 42


In [ ]:
# ── 7.1  Isolation Forest ────────────────────────────────────────────────────
ANOMALY_FEATURES = ['Revenue', 'Units', 'Margin_%', 'Selling_Price',
                    'Cost_Price', 'Stock_On_Hand', 'Reorder_Level', 'Lead_Time_Days']

ai_df     = df[ANOMALY_FEATURES].dropna().copy()
valid_idx = ai_df.index

scaler_if = StandardScaler()
X_if      = scaler_if.fit_transform(ai_df.values)

iso_forest = IsolationForest(n_estimators=200, contamination=0.03, random_state=42, n_jobs=-1)
if_preds   = iso_forest.fit_predict(X_if)
if_scores  = iso_forest.decision_function(X_if)

df_ai             = df.loc[valid_idx].copy()
df_ai['IF_Pred']  = if_preds
df_ai['IF_Score'] = if_scores
df_ai['Anomaly']  = df_ai['IF_Pred'].map({-1: 'Anomaly', 1: 'Normal'})

n_anomalies    = (df_ai['IF_Pred'] == -1).sum()
n_normal       = (df_ai['IF_Pred'] ==  1).sum()
pct_anom       = n_anomalies / len(df_ai) * 100
anom_rev_mean  = df_ai[df_ai['IF_Pred'] == -1]['Revenue'].mean()
norm_rev_mean  = df_ai[df_ai['IF_Pred'] ==  1]['Revenue'].mean()
anom_marg_mean = df_ai[df_ai['IF_Pred'] == -1]['Margin_%'].mean() * 100
norm_marg_mean = df_ai[df_ai['IF_Pred'] ==  1]['Margin_%'].mean() * 100

print('=' * 55)
print(f'  ⚠️  Anomalies detected    : {n_anomalies:,}  ({pct_anom:.1f}%)')
print(f'  ✅  Normal transactions  : {n_normal:,}  ({100-pct_anom:.1f}%)')
print(f'  💰  Anomaly avg revenue  : {fmt_inr(anom_rev_mean)}')
print(f'  💰  Normal  avg revenue  : {fmt_inr(norm_rev_mean)}')
print(f'  📐  Anomaly avg margin % : {anom_marg_mean:.2f}%')
print(f'  📐  Normal  avg margin % : {norm_marg_mean:.2f}%')
print('=' * 55)


In [ ]:
# ── 7.2  Anomaly Score Distribution ──────────────────────────────────────────
normal_scores = df_ai[df_ai['Anomaly'] == 'Normal']['IF_Score']
anom_scores   = df_ai[df_ai['Anomaly'] == 'Anomaly']['IF_Score']

fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(normal_scores, bins=60, color='#10b981', alpha=0.7, label='Normal')
ax.hist(anom_scores,   bins=60, color='#ef4444', alpha=0.7, label='Anomaly')
ax.axvline(x=0, color='#f59e0b', linewidth=2, linestyle='--', label='Decision boundary (0)')
ax.set_xlabel('Isolation Forest Score  (more negative = more anomalous)')
ax.set_ylabel('Frequency')
ax.set_title('Anomaly Score Distribution')
ax.legend(facecolor='#1e1e2e', edgecolor='#2e2e4e', labelcolor='#c8cfe8')
plt.tight_layout()
plt.show()


In [ ]:
# ── 7.3  Revenue vs Margin % — Anomalies Highlighted ────────────────────────
sample   = df_ai.sample(n=min(8000, len(df_ai)), random_state=42)
normal_s = sample[sample['Anomaly'] == 'Normal']
anom_s   = sample[sample['Anomaly'] == 'Anomaly']

fig, ax = plt.subplots(figsize=(12, 6))
ax.scatter(normal_s['Revenue'], normal_s['Margin_%'], s=4,
           color='#3b82d4', alpha=0.4, label=f'Normal ({n_normal:,})')
ax.scatter(anom_s['Revenue'],   anom_s['Margin_%'],   s=10,
           color='#ef4444', alpha=0.75, label=f'Anomaly ({n_anomalies:,})')
ax.set_xlabel('Revenue (₹)')
ax.set_ylabel('Margin % (ratio)')
ax.set_title('Revenue vs Margin % — Anomalies Highlighted')
ax.legend(facecolor='#1e1e2e', edgecolor='#2e2e4e', labelcolor='#c8cfe8')
plt.tight_layout()
plt.show()


In [ ]:
# ── 7.4  Anomaly Count by Category & Brand ───────────────────────────────────
anom_cat   = df_ai[df_ai['IF_Pred'] == -1].groupby('Category').size().sort_values()
anom_brand = df_ai[df_ai['IF_Pred'] == -1].groupby('Brand').size().sort_values()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.barh(anom_cat.index, anom_cat.values, color='#ef4444')
for i, val in enumerate(anom_cat.values):
    ax1.text(val + 1, i, str(val), va='center', fontsize=9, color='#c8cfe8')
ax1.set_xlabel('Anomaly Count')
ax1.set_title('Anomaly Count by Category')

ax2.barh(anom_brand.index, anom_brand.values, color='#f59e0b')
for i, val in enumerate(anom_brand.values):
    ax2.text(val + 1, i, str(val), va='center', fontsize=9, color='#c8cfe8')
ax2.set_xlabel('Anomaly Count')
ax2.set_title('Anomaly Count by Brand')

plt.tight_layout()
plt.show()


In [ ]:
# ── 7.5  Top 25 Most Anomalous Transactions ──────────────────────────────────
top_anom = (
    df_ai[df_ai['IF_Pred'] == -1]
    .nsmallest(25, 'IF_Score')[[
        'Invoice_ID', 'Brand', 'Category', 'City', 'Channel',
        'Units', 'Revenue', 'Margin_%', 'Stock_On_Hand',
        'Reorder_Level', 'Lead_Time_Days', 'IF_Score'
    ]].copy()
)
top_anom['Revenue']  = top_anom['Revenue'].map(fmt_inr)
top_anom['Margin_%'] = top_anom['Margin_%'].map(lambda v: f'{v*100:.1f}%')
top_anom['IF_Score'] = top_anom['IF_Score'].map(lambda v: f'{v:.4f}')
top_anom.index = range(1, len(top_anom) + 1)
print('Top 25 Most Anomalous Transactions (sorted by IF_Score ascending):')
top_anom


### 7.2 K-Means Clustering — Transaction Segmentation

**Features:** Revenue, Units, Margin_%, Selling_Price, Stock_On_Hand, Lead_Time_Days  
**k = 4 clusters**  ·  n_init = 20  ·  StandardScaler applied  
**PCA** (2 components) used for visualisation only


In [ ]:
# ── 7.6  K-Means Clustering ───────────────────────────────────────────────────
CLUSTER_FEATURES = ['Revenue', 'Units', 'Margin_%', 'Selling_Price',
                    'Stock_On_Hand', 'Lead_Time_Days']
N_CLUSTERS = 4

cl_df  = df[CLUSTER_FEATURES].dropna().copy()
cl_idx = cl_df.index

scaler_km = StandardScaler()
X_km      = scaler_km.fit_transform(cl_df.values)

km     = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=20)
labels = km.fit_predict(X_km)

pca    = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_km)

print(f'K-Means inertia         : {km.inertia_:,.0f}')
print(f'PCA variance explained  : PC1={pca.explained_variance_ratio_[0]*100:.1f}%  PC2={pca.explained_variance_ratio_[1]*100:.1f}%')
print(f'Total explained         : {sum(pca.explained_variance_ratio_)*100:.1f}%')


In [ ]:
# ── 7.7  Cluster Profiles ────────────────────────────────────────────────────
df_cl           = df.loc[cl_idx].copy()
df_cl['Cluster'] = labels.astype(str)
df_cl['PCA_1']   = coords[:, 0]
df_cl['PCA_2']   = coords[:, 1]

cluster_summary = (
    df_cl.groupby('Cluster')
    .agg(
        Count         =('Revenue', 'count'),
        Avg_Revenue   =('Revenue', 'mean'),
        Avg_Margin_Pct=('Margin_%', 'mean'),
        Avg_Units     =('Units', 'mean'),
        Avg_Sell_Price=('Selling_Price', 'mean'),
        Avg_Stock     =('Stock_On_Hand', 'mean'),
    )
    .sort_values('Avg_Revenue', ascending=False)
    .reset_index()
)

segment_names = ['Premium / High-Value', 'Mid-Tier Growth',
                 'Budget / High-Volume', 'Low-Activity Tail']
cluster_summary['Segment'] = segment_names
name_map     = dict(zip(cluster_summary['Cluster'], cluster_summary['Segment']))
df_cl['Segment'] = df_cl['Cluster'].map(name_map)

profile = cluster_summary.copy()
profile['Avg_Revenue']    = profile['Avg_Revenue'].map(fmt_inr)
profile['Avg_Margin_Pct'] = profile['Avg_Margin_Pct'].map(lambda v: f'{v*100:.1f}%')
profile['Avg_Units']      = profile['Avg_Units'].map(lambda v: f'{v:.1f}')
profile['Avg_Sell_Price'] = profile['Avg_Sell_Price'].map(lambda v: f'₹{v:.1f}')
profile['Avg_Stock']      = profile['Avg_Stock'].map(lambda v: f'{v:.0f}')
profile['Count']          = profile['Count'].map(lambda v: f'{v:,}')
profile.index = range(1, len(profile) + 1)
print('Cluster Profiles (sorted by Avg Revenue descending):')
profile[['Segment', 'Count', 'Avg_Revenue', 'Avg_Margin_Pct', 'Avg_Units', 'Avg_Sell_Price', 'Avg_Stock']]


In [ ]:
# ── 7.8  PCA Cluster Scatter ──────────────────────────────────────────────────
seg_palette = {
    'Premium / High-Value': '#7c5cd8',
    'Mid-Tier Growth'     : '#3b82d4',
    'Budget / High-Volume': '#10b981',
    'Low-Activity Tail'   : '#f59e0b',
}
pca_sample = df_cl.sample(n=min(6000, len(df_cl)), random_state=42)

fig, ax = plt.subplots(figsize=(11, 7))
for seg, grp in pca_sample.groupby('Segment'):
    ax.scatter(grp['PCA_1'], grp['PCA_2'], s=6, alpha=0.55,
               color=seg_palette[seg], label=seg)
ax.set_xlabel('PCA Component 1')
ax.set_ylabel('PCA Component 2')
ax.set_title(
    f'Transaction Clusters — PCA Projection  '
    f'({sum(pca.explained_variance_ratio_)*100:.1f}% variance explained)'
)
ax.legend(facecolor='#1e1e2e', edgecolor='#2e2e4e', labelcolor='#c8cfe8', markerscale=3)
plt.tight_layout()
plt.show()


In [ ]:
# ── 7.9  Segment Transaction Count + Category Mix ────────────────────────────
seg_counts = df_cl['Segment'].value_counts()
cat_seg    = df_cl.groupby(['Segment', 'Category']).size().unstack(fill_value=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

bars = ax1.bar(seg_counts.index, seg_counts.values,
               color=[seg_palette.get(s, '#7c5cd8') for s in seg_counts.index])
for bar, val in zip(bars, seg_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, val * 1.005,
             f'{val:,}', ha='center', fontsize=8, color='#c8cfe8')
ax1.set_title('Transaction Count by Segment')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=15)

cat_seg.plot(kind='bar', ax=ax2, stacked=True, color=PALETTE, edgecolor='#0e0e1a')
ax2.set_title('Category Mix by Segment')
ax2.set_ylabel('Transactions')
ax2.tick_params(axis='x', rotation=15)
ax2.legend(facecolor='#1e1e2e', edgecolor='#2e2e4e', labelcolor='#c8cfe8',
           fontsize=8, loc='upper right')

plt.tight_layout()
plt.show()


<a id='8'></a>
## 8. Key Business Findings

In [ ]:
# ── 8. Summary of All Key Business Findings ─────────────────────────────────
top_anom_cat   = df_ai[df_ai['IF_Pred'] == -1]['Category'].value_counts().idxmax()
top_anom_brand = df_ai[df_ai['IF_Pred'] == -1]['Brand'].value_counts().idxmax()
top_seg_name   = cluster_summary.iloc[0]['Segment']
bot_seg_name   = cluster_summary.iloc[-1]['Segment']

findings = [
    ('SALES & PROFITABILITY', [
        f'Total Revenue             : {fmt_inr(total_rev)}',
        f'Total Margin              : {fmt_inr(total_margin)}',
        f'Avg Margin %              : {avg_margin_pct:.2f}%',
        'Revenue scales strongly with units sold (r = 0.63).',
        'Margin % is independent of basket size (r = 0.12).',
        'All 8 brands maintain positive margins throughout 2024.',
    ]),
    ('INVENTORY RISK', [
        f'{reorder_req:,} transactions ({reorder_req/len(df)*100:.1f}%) carry Reorder Required status.',
        'Lead time range: 3 – 14 days.  Near-zero correlation with revenue.',
        'Inventory planning and demand performance are managed independently.',
    ]),
    ('CUSTOMER BEHAVIOUR', [
        f'Loyal customers account for {loyal_pct:.1f}% of transactions.',
        f'Loyal revenue  : {fmt_inr(loyal_rev)}  |  Non-loyal: {fmt_inr(nonloyal_rev)}',
        'Customer age has near-zero correlation with revenue (r ≈ 0.006).',
        '40.1% of Customer_Age values were missing (imputed with median).',
    ]),
    ('AI FINDINGS — ISOLATION FOREST', [
        f'{n_anomalies:,} anomalies detected ({pct_anom:.1f}%) out of {len(df_ai):,} transactions.',
        f'Top anomaly category: {top_anom_cat}  |  Top anomaly brand: {top_anom_brand}',
        f'Anomaly avg Margin %: {anom_marg_mean:.2f}%  vs  Normal: {norm_marg_mean:.2f}%',
        'Flagged records deviate across revenue, margin, pricing, and inventory simultaneously.',
    ]),
    ('AI FINDINGS — K-MEANS CLUSTERING', [
        f'4 natural transaction segments identified.',
        f"Top segment: '{top_seg_name}' — highest avg revenue per transaction.",
        f"Low segment: '{bot_seg_name}' — target with promotions or bundle pricing.",
        f'PCA explains {sum(pca.explained_variance_ratio_)*100:.1f}% of variance in 2 components.',
    ]),
    ('RECOMMENDED ACTIONS', [
        'Audit top 25 anomalous transactions with finance/operations teams.',
        "Prioritise 'Premium / High-Value' cluster for loyalty retention programmes.",
        'Investigate reorder-required cities to prevent stock-outs.',
        'Review brand-level margin anomalies for discount abuse or cost escalation.',
    ]),
]

for section, points in findings:
    print(f'\n{"=" * 60}')
    print(f'  {section}')
    print(f'  {"-" * 56}')
    for p in points:
        print(f'    • {p}')
print(f'\n{"=" * 60}')
print('\n✅  FMCG InsightAI — Analysis Complete')
print('    Author : Manthan Tighare')
print('    Dataset: Indian FMCG Retail Sales, Customer & Inventory (2024)')
